# 07.3 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/07-forecasting/03-smoothing-holt-winters/exercises.ipynb)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

y = [100, 120, 110, 130, 125]      # £k, five weeks

def ses_by_hand(y, alpha):
    level = y[0]                    # start the level at the first value
    rows = []
    for t in range(1, len(y)):
        forecast = level            # forecast for week t is the level so far
        level = alpha * y[t] + (1 - alpha) * level
        rows.append({"week": t + 1, "actual": y[t], "forecast": forecast,
                     "error": y[t] - forecast, "new level": level})
    return pd.DataFrame(rows)

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

# The two sheets overlap: "Year 2009-2010" runs to 9 Dec 2010, "Year 2010-2011" starts 1 Dec 2010.
older = load_retail("Year 2009-2010")
newer = load_retail("Year 2010-2011")
raw = pd.concat([older[older["InvoiceDate"] < "2010-12-01"], newer], ignore_index=True)

def clean_retail(df):
    """The standard ShopWave cleaning used from lesson 01.5 onwards."""
    out = df[~df["Invoice"].astype(str).str.startswith("C")]
    out = out[(out["Quantity"] > 0) & (out["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
    return out.assign(Revenue=out["Quantity"] * out["Price"])

sales = clean_retail(raw)
sales = sales[sales["Quantity"] < 70_000]          # two one-off orders, both cancelled the same day
END = pd.Timestamp("2011-12-04")                    # last complete week (Mon 28 Nov - Sun 4 Dec)
sales = sales[sales["InvoiceDate"] < END + pd.Timedelta(days=1)]

rev = sales.set_index("InvoiceDate")["Revenue"]
daily = rev.resample("D").sum()                     # every calendar day, 0 when closed
weekly = rev.resample("W").sum()                    # weeks end on Sunday
monthly = rev.resample("ME").sum()                  # month-end dates
monthly = monthly[monthly.index <= "2011-11-30"]    # December 2011 has only 4 days: drop it
print(f"{len(sales):,} order lines, {daily.index[0].date()} to {daily.index[-1].date()}")
print(f"{len(daily)} days, {len(weekly)} weeks, {len(monthly)} months")

H = 5                                                   # forecast 5 weeks ahead
ORIGINS = pd.date_range("2011-04-03", periods=8, freq="4W-SUN")      # 3 Apr ... 16 Oct 2011

def backtest(forecast_fn, series=None, origins=ORIGINS, h=H):
    """Rolling-origin backtest: at each origin, fit on the past, forecast h weeks, score MAE."""
    series = weekly if series is None else series
    rows = []
    for origin in origins:
        train = series[series.index <= origin]
        test = series[series.index > origin].iloc[:h]
        pred = np.asarray(forecast_fn(train, len(test)))
        rows.append({"origin": origin.date(), "MAE": np.abs(test.values - pred).mean()})
    return pd.DataFrame(rows)

def naive(train, h):          return np.repeat(train.iloc[-1], h)
def mean_fc(train, h):        return np.repeat(train.mean(), h)
def seasonal_naive(train, h): return train.iloc[-52:-52 + h].values
def drift(train, h):
    slope = (train.iloc[-1] - train.iloc[0]) / (len(train) - 1)
    return train.iloc[-1] + slope * np.arange(1, h + 1)

from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing

def ses(train, h):
    return SimpleExpSmoothing(train, initialization_method="estimated").fit().forecast(h).values

def holt(train, h):
    return ExponentialSmoothing(train, trend="add", damped_trend=True,
                                initialization_method="estimated").fit().forecast(h).values

from statsmodels.tsa.holtwinters import ExponentialSmoothing

def holt_winters(train, h):
    """Additive Holt-Winters, seeded with the first year as the starting seasonal pattern."""
    first = train.iloc[:52]
    model = ExponentialSmoothing(
        train, trend="add", damped_trend=True, seasonal="add", seasonal_periods=52,
        initialization_method="known", initial_level=first.mean(), initial_trend=0.0,
        initial_seasonal=(first - first.mean()).values).fit()
    return model.forecast(h).values

## Exercise 1: by hand

With α = 0.3 and starting level 50, run SES by hand on `[50, 60, 55, 70]`. What is the forecast for the fifth value? Check with `ses_by_hand`.

In [ ]:
# your code here

## Exercise 2: choosing α

On the weekly series up to 16 October 2011, fit SES with α fixed at 0.1, 0.3, 0.5, 0.7 and 0.9 (`fit(smoothing_level=a, optimized=False)`). For each, report the in-sample MAE of the one-step forecasts (`fittedvalues`). Which α fits best? Does it match the α statsmodels chose?

In [ ]:
# your code here

## Exercise 3: undamped Holt

Backtest Holt **without** damping (`damped_trend=False`). Is it better or worse than damped Holt? Look at one forecast and explain.

In [ ]:
# your code here

## Exercise 4: monthly Holt-Winters

Monthly data has a 12-month season and never hits zero. Fit a **multiplicative** Holt-Winters on `monthly` up to October 2011 using the first 12 months as the known seasonal start (`initial_seasonal = first / first.mean()`), and forecast November 2011. How close is it to the actual £1.16m?

In [ ]:
# your code here